In [1]:
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from hydra import compose, initialize
from src.utils import compute_item_distance_matrix

In [ ]:
import torch


class ILDMetric:
    """
    Vectorized ILD:
    ILD(u) = mean over i<j of dist(item_i, item_j)
    """

    def __init__(self, k, dist_matrix):
        self._k = k
        self._dist = dist_matrix  # (num_items, num_items)
        self._triu = torch.triu_indices(k, k, offset=1)  # (2, K(K - 1)/2)

    @classmethod
    def create_from_config(cls, config, **kwargs):
        return cls(
            k=config["k"],
            dist_matrix=kwargs["dist_matrix"],
        )

    def __call__(self, inputs):
        """
        inputs["logits"]: (B, K) item ids
        Returns: tensor (B,) — ILD per user
        """
        topk = inputs["logits"][:, : self._k]  # (B, K)

        sub = self._dist[topk[:, None], topk[:, :, None]]  # (B, (K, K))

        pairwise = sub[:, self._triu[0], self._triu[1]]  # (B, K(K - 1)/2)

        ild_per_user = pairwise.mean(dim=1)  # (B,)

        return ild_per_user

    def reduce(self, ild_per_user):
        """
        ild_per_user: tensor (B,)
        """
        return sum(ild_per_user) / len(ild_per_user)

In [3]:
from pathlib import Path
import json

import hydra
from hydra.core.hydra_config import HydraConfig
import pandas as pd
from omegaconf import OmegaConf
from torch import nn
from torch.utils.data import DataLoader

from src.dataset import SequenceDataset, build_graph
from src.loss import LocalObjective, MRGSRecLoss
from src.metrics import CoverageMetric, NDCGMetric, RecallMetric, StatefullMetric
from src.model import MRGSRecModel
from src.optimizer import BasicOptimizer
from src.sequence import SequentialEncoder
from src.utils import (
    BasicBatchProcessor,
    create_logger,
    fix_random_seed,
    train,
    save_metrics,
)

logger = create_logger(name=__name__)
seed_val = 42


def unpack_dataset(dataset_config):
    data_folder = Path(dataset_config["path_to_data_dir"])
    dataset_name = dataset_config["name"]
    all_data = pd.read_csv(data_folder / f"{dataset_name}.csv")
    split_folder = data_folder / "global_split" / dataset_name
    train_path = split_folder / "train.csv"
    val_path = split_folder / "validation.csv"
    test_path = split_folder / "test.csv"

    return all_data, train_path, val_path, test_path

In [4]:
with initialize(config_path="configs"):
    cfg = compose(config_name="beauty_sasrec")

/var/folders/h4/dkxs7qks1g9ckyjx2kx5_y680000gn/T/ipykernel_29989/977067326.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="configs"):
[2025-11-24 18:35:25] [DEBUG]: Setting JobRuntime:name=UNKNOWN_NAME
[2025-11-24 18:35:25] [DEBUG]: Setting JobRuntime:name=notebook
/Users/arturgimranov/CS/work/multirepr_recsys/.venv/lib/python3.11/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'beauty_sasrec': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [5]:
cfg["device"] = "cpu"
cfg["num_epochs"] = 2

In [6]:
model_name = cfg["model_name"]
assert model_name in ["sasrec", "mrgsrec"]
fix_random_seed(seed_val)
config = OmegaConf.to_container(cfg, resolve=True)
config["model"]["topk_k"] = max(config["metrics_ks"])
device = config["device"]

logger.info(f"Training config:\n{OmegaConf.to_yaml(config)}\n")
logger.info(f"Current DEVICE: {device}")

# TODO: dumb a little
all_data, train_path, val_path, test_path = unpack_dataset(config["dataset"])

dataset_meta = {
    "num_users": all_data["user_id"].max(),
    "num_items": all_data["item_id"].max(),
    "max_sequence_length": config["dataset"]["max_sequence_length"],
}

train_sampler = SequenceDataset(
    train_path,
    config["dataset"]["max_sequence_length"],
    mode="train",
)

validation_sampler = SequenceDataset(
    val_path,
    config["dataset"]["max_sequence_length"],
    mode="test",
    all_data=all_data,
)

test_sampler = SequenceDataset(
    test_path,
    config["dataset"]["max_sequence_length"],
    mode="test",
)

collator = BasicBatchProcessor()
train_dataloader = DataLoader(
    dataset=train_sampler, **config["dataloader"]["train"], collate_fn=collator
)
validation_dataloader = DataLoader(
    dataset=validation_sampler,
    **config["dataloader"]["validation"],
    collate_fn=collator,
)
test_dataloader = DataLoader(
    dataset=test_sampler, **config["dataloader"]["validation"], collate_fn=collator
)

[2025-11-24 18:35:25] [INFO]: Training config:
dataloader:
  train:
    batch_size: 128
    drop_last: false
    shuffle: true
  validation:
    batch_size: 256
    drop_last: false
    shuffle: false
loss:
  local:
    predictions_prefix: local_prediction
    labels_prefix: positive
  global:
    positive_prefix: global_positive
    negative_prefix: global_negative
  fusion:
    positive_prefix: fusion_positive
    negative_prefix: fusion_negative
  contrastive:
    fst_embeddings_prefix: contrastive_fst_embeddings
    snd_embeddings_prefix: contrastive_snd_embeddings
device: cpu
num_epochs: 2
early_stopping_rounds: 50
metrics_ks:
- 10
- 100
dataset:
  path_to_data_dir: data
  name: reviews_Beauty_5
  max_sequence_length: 10
model:
  embedding_dim: 128
  num_heads: 1
  num_layers: 4
  dim_feedforward: 128
  dropout: 0.4
  activation: gelu
  layer_norm_eps: 1.0e-09
  topk_k: 100
optimizer:
  optimizer:
    type: adam
    lr: 0.0014002963516611
  clip_grad_threshold: 10.0
model_name: sa

In [7]:
all_data.head()

,user_id,item_id,rating,timestamp
0,225,104,1,1023840000
1,225,101,1,1024185600
2,225,102,1,1024185600
3,1115,152,1,1036627200
4,225,23,1,1052611200


In [8]:
item_distances = compute_item_distance_matrix(
    all_data, dataset_meta["num_items"] + 2, dataset_meta["num_users"] + 2
)

In [9]:
_embedding_dim = config["model"]["embedding_dim"]
_num_users = dataset_meta["num_users"]
_num_items = dataset_meta["num_items"]
_max_sequence_length = dataset_meta["max_sequence_length"]

item_embeddings = nn.Embedding(
    num_embeddings=_num_items + 2,
    embedding_dim=_embedding_dim,
    padding_idx=0,
)
position_embeddings = nn.Embedding(
    num_embeddings=_max_sequence_length
    + 1,  # in order to include `max_sequence_length` value
    embedding_dim=_embedding_dim,
)
model = SequentialEncoder(
    **config["model"],
    position_embeddings=position_embeddings,
    item_embeddings=item_embeddings,
    num_items=_num_items,
).to(device)
loss_function = LocalObjective()
optimizer = BasicOptimizer.create_from_config(config["optimizer"], model=model)

/Users/arturgimranov/CS/work/multirepr_recsys/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(


In [10]:
item2num_iteractions = all_data.groupby("item_id").count()["user_id"].to_dict()

In [11]:
metrics = {}
for k in config["metrics_ks"]:
    metrics |= {
        f"ndcg@{k}": NDCGMetric(k),
        f"coverage@{k}": CoverageMetric(k, dataset_meta["num_items"]),
        f"recall@{k}": RecallMetric(k),
        f"diversity@{k}": ILDMetric(
            k,
            item_distances,
        ),
    }

inference_dict_validation = dict(
    dataloader=validation_dataloader,
    model=model,
    metrics=metrics,
    device=device,
)

inference_dict_test = dict(
    dataloader=test_dataloader,
    model=model,
    metrics=metrics,
    device=device,
)

# Train process
all_metrics_list, metrics = train(
    dataloader=train_dataloader,
    model=model,
    optimizer=optimizer,
    loss_function=loss_function,
    num_epochs=config["num_epochs"],
    early_stopping_rounds=config["early_stopping_rounds"],
    device=device,
    best_metric=config.get("best_metric"),
    inference_dict_validation=inference_dict_validation,
    inference_dict_test=inference_dict_test,
)

[2025-11-24 18:35:29] [DEBUG]: Start training...
[2025-11-24 18:35:29] [DEBUG]: Start epoch 0
Epoch 0: 100%|██████████| 145/145 [00:07<00:00, 18.62it/s]


VAL
Inference procedure has been finished!
Metrics are the following:
ndcg@10: 0.0022265746164574192
coverage@10: 0.013883150152879928
recall@10: 0.004867724867724868
diversity@10: 0.9744946956634521
ndcg@100: 0.007164998246879174
coverage@100: 0.04776464754978928
recall@100: 0.030264550264550265
diversity@100: 0.9928961992263794
Metrics finished!
TEST


[2025-11-24 18:35:38] [DEBUG]: Start epoch 1


Inference procedure has been finished!
Metrics are the following:
ndcg@10: 0.0013666293274450475
coverage@10: 0.01586645731757706
recall@10: 0.0033577913194876258
diversity@10: 0.9745646715164185
ndcg@100: 0.005889735885215568
coverage@100: 0.05544996281299066
recall@100: 0.027857231687601045
diversity@100: 0.9929267168045044
Metrics finished!


Epoch 1: 100%|██████████| 145/145 [00:07<00:00, 19.08it/s]


VAL
Inference procedure has been finished!
Metrics are the following:
ndcg@10: 0.0032937350096525968
coverage@10: 0.008015866457317578
recall@10: 0.008042328042328042
diversity@10: 0.9646444916725159
ndcg@100: 0.008895281985323265
coverage@100: 0.0411536236674655
recall@100: 0.037037037037037035
diversity@100: 0.9819134473800659
Metrics finished!
TEST
Inference procedure has been finished!
Metrics are the following:
ndcg@10: 0.002144124185159838
coverage@10: 0.00851169324849186
recall@10: 0.004352692451187664
diversity@10: 0.9646403789520264
ndcg@100: 0.006426737307316183
coverage@100: 0.04718618296008594
recall@100: 0.02636487999005099
diversity@100: 0.9820181727409363
Metrics finished!
Total time: 18.009984016418457
